# Evaluation results — F1 summary

Reads an `evaluation_results.jsonl` (one JSON object per evaluated image) and reports:

1. **The execution summary** — the `Avg F1 (mean)` and `Avg F1 (median)` figures the pipeline
   prints at the end of a run, plus the standard deviation it omits.
2. **F1 per extraction field**, broken down by document type
   (`BANK_STATEMENT`, `INVOICE`, `RECEIPT`), as a macro table and a micro table.

Set `RESULTS_PATH` below to the run you want. For the production `synthetic_21` run that
reported 0.807 / 0.825 over 21 images, that is the `evaluation_results.jsonl` under its output
directory (`/efs/shared/annotations/synthetic_21/evaluation`) — copy it locally first.

## Which average?

The three numbers here are all "mean F1" and all differ. They are not competing estimates of one
quantity; they answer different questions:

| | unit of account | computed as |
|---|---|---|
| execution summary | one document | mean over documents of a per-document field aggregate |
| macro field table | one document | mean over documents of that field's F1 |
| micro field table | one extracted item | F1 of the pooled tp/fp/fn for that field |

For single-valued fields (`DOCUMENT_TYPE`, `STATEMENT_DATE_RANGE`) macro and micro coincide. For
list-valued fields they diverge, because micro lets a 31-transaction statement outweigh a
one-line receipt 31 to 1. Always say which one you are quoting.

In [ ]:
from pathlib import Path
import json

import pandas as pd

# Point this at the results file you want to analyse.
RESULTS_PATH = Path("/path/to/evaluation_results.jsonl")

In [ ]:
def read_records(path: Path) -> list[dict]:
    """Read the scored records from an evaluation_results.jsonl.

    Applies the same filter as the production summary (stages/evaluate.py:191): a record
    counts only if it carries a median_f1 and has no error.

    Args:
        path: Path to the JSONL results file.

    Returns:
        The scored records, in file order.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"No results file at {path}.\n"
            "Set RESULTS_PATH in the cell above to your run's evaluation_results.jsonl, e.g.\n"
            '    RESULTS_PATH = Path("~/evaluation_data/output/evaluation_results.jsonl").expanduser()\n'
            "The evaluate stage writes it to its output directory — the 'Output Directory' row "
            "of the pipeline's execution summary. Copy it locally if the run was remote."
        )

    records, skipped = [], []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if record.get("error") or "median_f1" not in record:
                skipped.append(record.get("image_name", "<unnamed>"))
                continue
            records.append(record)
    if not records:
        raise ValueError(f"No scored records (median_f1 present, no error) found in {path}")
    if skipped:
        print(f"Skipped {len(skipped)} unscored/errored record(s): {', '.join(skipped)}")
    return records


def field_frame(records: list[dict]) -> pd.DataFrame:
    """Flatten records into one row per (image, field).

    Args:
        records: Scored records from read_records.

    Returns:
        Long-format frame with image_name, document_type, field, f1_score, tp, fp, fn.
    """
    return pd.DataFrame(
        [
            {
                "image_name": record["image_name"],
                "document_type": record["document_type"],
                "field": field,
                "f1_score": field_score["f1_score"],
                "tp": field_score["tp"],
                "fp": field_score["fp"],
                "fn": field_score["fn"],
            }
            for record in records
            for field, field_score in record["field_scores"].items()
        ]
    )


def document_frame(records: list[dict]) -> pd.DataFrame:
    """Build one row per document with the two per-document F1 aggregates.

    doc_mean_f1 is the record's overall_accuracy (mean of its field F1 scores) and
    doc_median_f1 is its median_f1. Both are recomputed from field_scores and checked
    against the stored values, so a schema change cannot silently skew the summary.

    Args:
        records: Scored records from read_records.

    Returns:
        Frame with image_name, document_type, n_fields, doc_mean_f1, doc_median_f1.
    """
    rows = []
    for record in records:
        f1_values = pd.Series([s["f1_score"] for s in record["field_scores"].values()])
        recomputed = {"doc_mean_f1": f1_values.mean(), "doc_median_f1": f1_values.median()}
        stored = {
            "doc_mean_f1": record["overall_accuracy"],
            "doc_median_f1": record["median_f1"],
        }
        for key, value in recomputed.items():
            if abs(value - stored[key]) > 1e-9:
                raise ValueError(
                    f"{key} disagrees with the stored value for {record['image_name']}: "
                    f"recomputed from field_scores = {value!r}, stored = {stored[key]!r}. "
                    "The record's field_scores no longer explain its headline metrics — "
                    "check the evaluator version that produced this file."
                )
        rows.append(
            {
                "image_name": record["image_name"],
                "document_type": record["document_type"],
                "n_fields": len(record["field_scores"]),
                **recomputed,
            }
        )
    return pd.DataFrame(rows)


records = read_records(RESULTS_PATH)
scores = field_frame(records)
documents = document_frame(records)

print(f"{len(documents)} documents, {scores['field'].nunique()} distinct fields")
print(documents["document_type"].value_counts().to_string())
documents.head()

## Execution summary — the two headline numbers, plus their spread

These reproduce the `Avg F1 (mean)` / `Avg F1 (median)` rows the pipeline prints at the end of
a run (`stages/evaluate.py:191-193`):

```python
scored        = [r for r in eval_results if "median_f1" in r and not r.get("error")]
avg_f1_mean   = sum(r["overall_accuracy"] for r in scored) / len(scored)
avg_f1_median = sum(r["median_f1"] for r in scored) / len(scored)
```

**Both rows are means across documents.** The `(mean)` / `(median)` in the label refers to how
the *fields within one document* were collapsed, not to how documents were combined:

* `Avg F1 (mean)` — mean over documents of each document's **mean** field F1 (`overall_accuracy`)
* `Avg F1 (median)` — mean over documents of each document's **median** field F1 (`median_f1`)

So `Avg F1 (median)` is not the median of anything at the corpus level. It reads higher than the
mean whenever a minority of fields fail badly, because the within-document median discards them —
that gap is the point of reporting both.

The pipeline prints only the two centres. The `std` column below is what it does not tell you:
the sample standard deviation (ddof=1) of the per-document scores.

In [ ]:
DOC_METRICS = {"Avg F1 (mean)": "doc_mean_f1", "Avg F1 (median)": "doc_median_f1"}


def execution_summary(docs: pd.DataFrame) -> pd.DataFrame:
    """Reproduce the pipeline's headline F1 rows and add their dispersion.

    Args:
        docs: Per-document frame from document_frame.

    Returns:
        One row per headline metric with n_images, mean, std, median, min and max
        taken across documents. The mean column is the value the pipeline prints.
    """
    summary = pd.DataFrame(
        {
            "n_images": docs[column].count(),
            "mean": docs[column].mean(),
            "std": docs[column].std(),
            "median": docs[column].median(),
            "min": docs[column].min(),
            "max": docs[column].max(),
        }
        for column in DOC_METRICS.values()
    )
    summary.index = pd.Index(DOC_METRICS.keys(), name="Metric")
    return summary


summary = execution_summary(documents)

# Printed the way the pipeline prints it, to 3 dp, so the two can be compared row by row.
for metric, row in summary.iterrows():
    print(f"{metric:<18} {row['mean']:.3f}   (sd {row['std']:.3f}, n={int(row['n_images'])})")

summary.round(4)

In [ ]:
# The same two headline metrics split by document type. The pipeline prints only the
# corpus-wide figure, so a run whose doc-type mix changes moves the headline for reasons
# that have nothing to do with model quality — this table shows which type is responsible.
summary_by_type = (
    documents.groupby("document_type")[list(DOC_METRICS.values())]
    .agg(["count", "mean", "std"])
    .round(4)
)
summary_by_type

## Three traps in the headline figures

These are properties of how the two numbers are constructed, not artefacts of one run. They all
push in the **optimistic** direction, so a headline read at face value overstates the model. The
cell below measures traps 1 and 2 on whichever file you loaded — the worked examples here come
from the 15-document bank-only run this notebook was developed against.

### 1. The scale does not start at zero

A document's `overall_accuracy` is the mean of its field F1 scores, and **each field counts once
regardless of how much work it represents**. A single date range and a 31-row transaction table
are worth the same 1/n.

That matters because some fields are effectively free. On the bank-statement schema, two of the
five — `DOCUMENT_TYPE` and `STATEMENT_DATE_RANGE` — score 1.0 on *every* document. `DOCUMENT_TYPE`
is not even an extraction: it is the classifier's label being marked against itself. So 0.4 of
the score is awarded before a single transaction is read:

> `doc_mean_f1` cannot fall below 0.400 on this schema; the worst document observed sits at 0.425.

The three fields doing the real work are compressed into the remaining 0.6 of the range. So a
headline is not "X% of the way to perfect". To rescale onto the range the model can actually
influence, use `(score - floor) / (1 - floor)`: this file's 0.887 becomes 0.81.

Do not assume the 0.4 floor transfers. It depends on the field schema and on which fields happen
to be saturated in *your* run, and for a mixed-document run it is a document-weighted blend of the
per-type floors. Take it from `floor_on_doc_mean_f1` below.

### 2. `Avg F1 (median)` is the *best* of the working fields, not the middle of anything

With five fields of which the top two are always 1.0, the sorted scores always look like
`[a, b, c, 1.0, 1.0]`. The median is the third value — `c` — which is the **maximum** of the three
transaction fields:

| document | sorted field F1 | median = |
|---|---|---|
| `westpac_debit_credit.png` | `[0.000, 0.062, 0.062, 1.0, 1.0]` | 0.062 — best of the three |
| `cba_home_loan.png` | `[0.400, 0.400, 0.857, 1.0, 1.0]` | 0.857 — best of the three |

So `median_f1` is a best-case-per-document statistic, and it reads exactly 1.0 for 11 of the 15
documents. Where `Avg F1 (median)` exceeds `Avg F1 (mean)`, that gap is not robustness — it is the
free fields propping up the middle while the two weakest extraction fields are discarded.

This too is a consequence of the field count and of *which* fields are saturated, so it does not
transfer automatically to invoices or receipts. The `median_equals_best_worked` column below
measures it per document type instead of assuming it.

### 3. The "median" row is the *less* stable of the two

Standard deviation across documents, from the table above: **0.172 for the mean row, 0.276 for the
median row** — the median is 60% noisier. This inverts the usual intuition that medians are
steadier, and it follows directly from trap 2: a max-of-three over a near-bimodal distribution is
a step function that snaps between 1.0 and near-zero, so it moves *more* between documents than an
average does.

Quoting 0.825 over 0.807 as the "safer, more robust" figure is therefore wrong twice over — it is
both the higher number and the noisier one.

In [ ]:
def scale_diagnostics(frame: pd.DataFrame, docs: pd.DataFrame) -> pd.DataFrame:
    """Quantify traps 1 and 2 for whatever file is loaded, per document type.

    Trap 1: a field scoring 1.0 on every document of its type adds a constant to every
    doc_mean_f1, putting a floor on the metric. The floor is that field count over the
    total field count.

    Trap 2: whether median_f1 collapses to "best of the fields that do real work" is
    decided empirically per document rather than by index arithmetic, so it stays correct
    for any field count or schema.

    Args:
        frame: Long-format frame from field_frame.
        docs: Per-document frame from document_frame.

    Returns:
        One row per document type: field counts, the implied floor, the worst observed
        score, how often the median equals the best worked field, and the free fields.
    """
    rows, index = [], []
    for doc_type, group in frame.groupby("document_type"):
        per_field_min = group.groupby("field")["f1_score"].min()
        free = set(per_field_min[per_field_min == 1.0].index)
        worked = group[~group["field"].isin(free)]

        best_worked = worked.groupby("image_name")["f1_score"].max()
        medians = docs.set_index("image_name")["doc_median_f1"].reindex(best_worked.index)
        equals_best = (medians - best_worked).abs() < 1e-9

        index.append(doc_type)
        rows.append(
            {
                "n_fields": len(per_field_min),
                "n_always_perfect": len(free),
                "floor_on_doc_mean_f1": len(free) / len(per_field_min),
                "worst_doc_mean_f1": docs.loc[docs["document_type"] == doc_type, "doc_mean_f1"].min(),
                "median_equals_best_worked": equals_best.mean(),
                "always_perfect_fields": ", ".join(sorted(free)) or "(none)",
            }
        )
    return pd.DataFrame(rows, index=pd.Index(index, name="document_type"))


diagnostics = scale_diagnostics(scores, documents)

for doc_type, row in diagnostics.iterrows():
    print(
        f"{doc_type}: {row.n_always_perfect}/{row.n_fields} fields perfect on every document"
        f" -> doc_mean_f1 floor {row.floor_on_doc_mean_f1:.3f}"
        f" (worst observed {row.worst_doc_mean_f1:.3f})"
    )
    if row.median_equals_best_worked > 0:
        print(
            f"    trap 2: median_f1 equals the BEST worked field on "
            f"{row.median_equals_best_worked:.0%} of these documents"
        )

diagnostics.round(4)

In [ ]:
MACRO_COLS = ["n_docs", "mean_f1", "std_f1", "min_f1", "max_f1"]
MICRO_COLS = ["n_items", "tp", "fp", "fn", "micro_precision", "micro_recall", "micro_f1"]


def summarise(frame: pd.DataFrame, *, by: list[str]) -> pd.DataFrame:
    """Aggregate macro and micro F1 statistics over the given grouping columns.

    Args:
        frame: Long-format frame from field_frame.
        by: Columns to group on, e.g. ["document_type", "field"].

    Returns:
        One row per group holding both the MACRO_COLS and MICRO_COLS statistics.
        std_f1 is the sample standard deviation (ddof=1) of the per-document F1
        scores, so it is NaN for any group holding a single document.
    """
    summary = frame.groupby(by, dropna=False).agg(
        n_docs=("image_name", "nunique"),
        mean_f1=("f1_score", "mean"),
        std_f1=("f1_score", "std"),
        min_f1=("f1_score", "min"),
        max_f1=("f1_score", "max"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum"),
    )
    summary["n_items"] = summary["tp"] + summary["fn"]
    predicted = summary["tp"] + summary["fp"]
    actual = summary["tp"] + summary["fn"]
    summary["micro_precision"] = (summary["tp"] / predicted).where(predicted > 0, 0.0)
    summary["micro_recall"] = (summary["tp"] / actual).where(actual > 0, 0.0)
    denominator = 2 * summary["tp"] + summary["fp"] + summary["fn"]
    summary["micro_f1"] = (2 * summary["tp"] / denominator).where(denominator > 0, 0.0)
    return summary[MACRO_COLS + MICRO_COLS]


overall = summarise(scores, by=["field"]).sort_values("mean_f1")
per_type = summarise(scores, by=["document_type", "field"]).sort_values(
    ["document_type", "mean_f1"]
)
per_type.shape, overall.shape

## Macro table — every document counts once

`mean_f1` is the average of the per-document F1 scores. This is the number to quote when you
want "how well does the model do on a typical document". It is insensitive to table length,
so a one-line receipt and a 31-transaction bank statement carry equal weight.

In [ ]:
macro_table = overall[MACRO_COLS].round(4)
macro_table.insert(
    1,
    "mean_pm_std",
    overall.apply(lambda r: f"{r.mean_f1:.3f} ± {r.std_f1:.3f}", axis=1),
)

macro_by_type = per_type[MACRO_COLS].round(4)

display(macro_table)
display(macro_by_type)

## Micro table — every extracted item counts once

`micro_f1` pools the raw `tp`/`fp`/`fn` counts across documents before computing the score, so
a 31-transaction statement weighs 31x a single-transaction one. This is the number to quote for
"what fraction of all transactions in the corpus did we get right". `n_items` is the ground-truth
item count (`tp + fn`) behind each row — check it before trusting a field's micro score.

Precision and recall are broken out because they fail differently: low recall means truncated
extraction, low precision means hallucinated rows.

In [ ]:
micro_table = overall[MICRO_COLS].sort_values("micro_f1").round(4)
micro_by_type = per_type[MICRO_COLS].sort_values(["document_type", "micro_f1"]).round(4)

display(micro_table)
display(micro_by_type)

In [ ]:
# Mean F1 (and its spread) as a field x document-type matrix.
# NaN = field not evaluated for that document type.
matrix = per_type["mean_f1"].unstack("document_type")
errors = per_type["std_f1"].unstack("document_type")
matrix.round(4)

In [ ]:
# Error bars are +/- 1 standard deviation across documents, so they clip outside [0, 1].
ax = matrix.plot.barh(
    figsize=(9, 0.45 * len(matrix) + 2),
    xlim=(0, 1),
    xerr=errors.fillna(0.0),
    capsize=3,
    error_kw={"elinewidth": 1, "ecolor": "0.3"},
)
ax.set_xlabel("mean F1 (± 1 sd)")
ax.set_ylabel("")
ax.set_title(f"Mean F1 per field — {RESULTS_PATH.name}")
ax.legend(title="document type", loc="lower right")
ax.grid(axis="x", alpha=0.3)

## Where the loss actually is

The micro table above separates precision from recall for a reason. In the 15-document run this
notebook was developed against, `micro_precision` is **exactly 1.0 on every field** while recall
runs 0.59–0.71. Check whether that still holds on your file, because it decides what to fix:

* **precision 1.0, recall low** — the model is *truncating*. It stops emitting transactions
  partway down the table and never invents one. The fix is in prompting, tiling or context
  length, not in output validation.
* **precision low** — the model is *hallucinating* rows, or the cleaner is mangling them. A very
  different investigation.

Collapsing these into a single `micro_f1` hides the distinction completely, which is why the
micro table is a separate table rather than one more column on the macro one.

The table below lists the worst (document, field) pairs. Expect the loss to be concentrated
rather than spread: in the reference run a single document contributed 31 of the missing items,
and removing it moves the headline more than any per-field prompt change would.

In [ ]:
# Worst documents per field — the usual starting point for error analysis.
scores.sort_values("f1_score").head(15)[
    ["document_type", "field", "image_name", "f1_score", "tp", "fp", "fn"]
]

## Quoting these numbers

A short decision list, because the tables above deliberately disagree with each other:

| You want to say | Use | From |
|---|---|---|
| "the run scored X" (matching the pipeline log) | `Avg F1 (mean)` | execution summary |
| "a typical document scores X on field F" | `mean_f1` | macro table |
| "we correctly extracted X% of all transactions" | `micro_f1` | micro table |
| "field F is unreliable" | `min_f1` + the worst-documents table | macro table |
| "the failure mode is truncation / hallucination" | `micro_recall` vs `micro_precision` | micro table |

Rules that follow from the traps above:

1. **Never quote a headline F1 without its `n`.** Both the document count and, for micro numbers,
   `n_items` change what the figure means.
2. **Never quote `Avg F1 (median)` as a robust or conservative statistic.** It is neither — it is
   the higher number and the noisier one. If you want robustness across documents, use the
   `median` column of the execution summary, which is a genuine corpus-level median.
3. **Never compare headlines across runs with different document-type mixes.** Compare the
   per-type tables, or a change in the mix will read as a change in quality.
4. **State the floor when quoting outside the team.** "0.887 on a scale whose floor is 0.400" is a
   different claim from "0.887 out of 1.0", and only the first is true.
5. **Do not report a corpus-level improvement without checking the worst documents.** The loss is
   concentrated, so a headline can move because one long statement got better while nothing else
   changed — or stay flat while a genuine fix is masked by one regression.